# Notion-Zotero Analysis

Full pipeline: load canonical bundles → build summary tables → clean → explore.

**Data sources** (pick one):
-  — bundles pulled via 
-  — bundles pulled via 
-  — bundles parsed from local fixture files

In [1]:
from pathlib import Path
import pandas as pd

from notion_zotero.analysis import (
    load_canonical_records,
    build_summary_dataframes,
    clean_table,
    run_analysis,
)

# Load from live pulled data
pulled_dir = Path("data/pulled/notion/learning_analytics_review")
if not pulled_dir.exists() or not any(pulled_dir.glob("*.canonical.json")):
    raise FileNotFoundError(
        f"No pulled data found in {pulled_dir}. "
        "Run: notion-zotero pull-notion"
    )

bundles = load_canonical_records(str(pulled_dir))
print(f"Loaded {len(bundles)} bundles from {pulled_dir}")

Loaded 442 bundles from data\pulled\notion\learning_analytics_review


In [2]:
# --- Cell 2: Inspect loaded bundles ---
print(f"Loaded {len(bundles)} canonical bundles")

if bundles:
    sample = bundles[0]
    refs = sample.get("references", [])
    if refs:
        r = refs[0]
        print(f"Sample: {r.get("title", "(no title)")} ({r.get("year", "?")})")

Loaded 442 canonical bundles
Sample: Contrastive Personalized Exercise Recommendation With Reinforcement Learning (2024)


In [3]:
# --- Cell 3: Build summary tables (accepted papers only) ---
def my_label_fn(task_name):
    mapping = {
        "performance_prediction": "PRED",
        "descriptive_modelling": "DESC",
        "recommender_systems": "REC",
        "knowledge_tracing": "KT",
    }
    key = (task_name or "").lower()
    return mapping.get(key, task_name)

def _status_signals(bundle):
    statuses = [w.get("state") for w in (bundle.get("workflow_states") or []) if w.get("state")]
    refs = bundle.get("references") or [{}]
    notion_props = (refs[0].get("sync_metadata") or {}).get("notion_properties") or {}
    statuses.extend(
        value for key, value in notion_props.items()
        if key.lower().startswith("status") and value
    )
    return [str(status) for status in statuses if str(status).strip()]

def _is_accepted(bundle):
    """Return True if any status signal is accepted (or no status info)."""
    statuses = _status_signals(bundle)
    if not statuses:
        return True  # no status info — include by default
    return any("accepted" in status.lower() for status in statuses)

accepted_bundles = [b for b in bundles if _is_accepted(b)]
print(f"Accepted: {len(accepted_bundles)} / {len(bundles)} total bundles")

dfs = build_summary_dataframes(accepted_bundles, task_label_fn=my_label_fn)
for name, df in dfs.items():
    print(f"  {name}: {len(df)} rows x {len(df.columns)} cols")
dfs.get("Reading List")

Accepted: 285 / 442 total bundles
  Reading List: 285 rows x 20 cols
  REC: 43 rows x 17 cols
  DESC: 79 rows x 20 cols
  PRED: 189 rows x 19 cols
  KT: 63 rows x 20 cols
  Summary Table: 2 rows x 25 cols


,id,title,authors,year,journal,doi,url,zotero_key,abstract,item_type,tags,search_terms,search_date,database,journal_quartile,provenance,validation_status,sync_metadata,Status,page_id
0,003e69cb-f908-470d-bab2-c26eed3c63d3,Contrastive Personalized Exercise Recommendati...,[Wu et al.],2024,IEEE Transactions on Learning Technologies,,,,,,[],“Recommender Systems” AND “Education”,2024-06-01,Google Scholar,Q2 Journal,{'source_id': '003e69cb-f908-470d-bab2-c26eed3...,unknown,{'notion_properties': {'Status': 'Accepted For...,Accepted For Recommender Systems,003e69cb-f908-470d-bab2-c26eed3c63d3
1,00f24000-f241-438b-a023-d872bc2b59e6,Using Clickstream Data Mining Techniques to Un...,[Rodriguez et al.],2021,LAK,,,,,,[],"""Learning Analytics” AND “Student Performance”",2023-01-01,ACM Digital Library,Conference,{'source_id': '00f24000-f241-438b-a023-d872bc2...,unknown,{'notion_properties': {'Status': 'Accepted for...,Accepted for Descriptive Modelling,00f24000-f241-438b-a023-d872bc2b59e6
2,02eedcf7-bc5a-406e-ba94-01f7e39a25cd,Enhancing educational evaluation through predi...,[Xuan Lam et al],2024,Computers & Education: Artificial Intelligence,,,,,,[],“Early Warning System” AND “Learning Analytics”,2024-04-02,Google Scholar,Q1 Journal,{'source_id': '02eedcf7-bc5a-406e-ba94-01f7e39...,unknown,{'notion_properties': {'Status': 'Accepted For...,Accepted For Performance Prediction Task,02eedcf7-bc5a-406e-ba94-01f7e39a25cd
3,02fef9f2-c5c2-47ee-a111-eb7788556c5e,Implementing AutoML in Educational Data Mining...,[Tsiakmaki et al.],2019,Applied Sciences,,,,,,[],“Learning Analytics” AND “Student Performance”,2023-01-01,Google Scholar,Q3+ Journal,{'source_id': '02fef9f2-c5c2-47ee-a111-eb77885...,unknown,{'notion_properties': {'Status': 'Accepted For...,Accepted For Performance Prediction Task,02fef9f2-c5c2-47ee-a111-eb7788556c5e
4,033342f8-04d7-4a83-b56e-7d8b056b12e2,A Robust Machine Learning Technique to Predict...,[Liao et al.],2019,ACM Transactions on Computing Education,,,,,,[],“Learning Management System” AND “Academic Suc...,2023-01-01,Google Scholar,Q1 Journal,{'source_id': '033342f8-04d7-4a83-b56e-7d8b056...,unknown,{'notion_properties': {'Status': 'Accepted For...,Accepted For Performance Prediction Task,033342f8-04d7-4a83-b56e-7d8b056b12e2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
280,fbabc039-d7a3-4330-88f4-1038cbba4145,Developing early warning systems to predict st...,[Hu et al.],2014,Computers in Human Behavior,,,,,,[],“Learning Analytics” AND “Student Performance”,2023-01-01,Google Scholar,Q1 Journal,{'source_id': 'fbabc039-d7a3-4330-88f4-1038cbb...,unknown,{'notion_properties': {'Status': 'Accepted For...,Accepted For Performance Prediction Task,fbabc039-d7a3-4330-88f4-1038cbba4145
281,fbe0c99f-8e17-471c-a700-6586193253e2,Early-predicting dropout of university student...,[Cannistrà et al.],2022,Studies in Higher Education,,,,,,[],"""Student Performance” AND “MAchine Learning”",2024-07-26,ISI Web of Science,Q1 Journal,{'source_id': 'fbe0c99f-8e17-471c-a700-6586193...,unknown,{'notion_properties': {'Status': 'Accepted For...,Accepted For Performance Prediction Task,fbe0c99f-8e17-471c-a700-6586193253e2
282,fd6153cf-0a1c-4664-9a99-74f4264d7d8e,A data-driven approach to predict first-year s...,[Gil et al.],2021,Education and Information Technologies,,,,,,[],“Learning Analytics” AND “Student Performance”,2023-01-01,Google Scholar,Q1 Journal,{'source_id': 'fd6153cf-0a1c-4664-9a99-74f4264...,unknown,{'notion_properties': {'Status': 'Accepted For...,Accepted For Performance Prediction Task,fd6153cf-0a1c-4664-9a99-74f4264d7d8e
283,fe76c297-6135-4a41-b537-d3637a4ac389,What are the most important predictors of comp...,[Hao et al.],2016,Computers in Human Behavior,,,,,,[],“Learning Strategies” AND “LMS”,2023-01-01,Google Scholar,Q1 Journal,{'source_id': 'fe76c297-6135-4a41-b537-d3637a4...,unknown,{'notion_properties': {'Status': 'Accepted for...,Accepted for Descriptive Modelling,fe76c297-6135-4a41-b537-d363

In [ ]:
# --- Cell 4: Clean tables (caller-provided fix dicts) ---
MY_TYPO_FIXES = {
    r"Fiiltering": "Filtering",
    r"Exerrcise": "Exercise",
    r"\bMAchine\b": "Machine",
}
MY_VALUE_MAP  = {"n/a": "Not Applicable", "none": "Not Applicable", "na": "Not Applicable"}
SEARCH_COLS   = ["search_terms", "Search Strategy"]

cleaned_dfs = {}
_clean_logs = {}
for name, df in dfs.items():
    cleaned, log = clean_table(df, typo_fixes=MY_TYPO_FIXES, value_map=MY_VALUE_MAP, search_strategy_columns=SEARCH_COLS)
    cleaned_dfs[name] = cleaned
    _clean_logs[name] = log
    print(f"  {name}: {log['text_updates']} cell(s) normalised")

In [5]:
# --- Cell 5: Normalization summary (accepted bundles, from Cell 3/4) ---
norm_log = {
    name: {
        "rows_before": len(dfs[name]),
        "rows_after": len(cleaned_dfs[name]),
        "text_updates": _clean_logs[name]["text_updates"],
        "search_strategy_updates": _clean_logs[name].get("search_strategy_updates", 0),
    }
    for name in dfs
}
pd.DataFrame(norm_log).T

,rows_before,rows_after,text_updates,search_strategy_updates
Reading List,285,285,30,0
REC,43,43,84,0
DESC,79,79,259,0
PRED,189,189,367,0
KT,63,63,178,0
Summary Table,2,2,6,0


In [6]:
# --- Cell 6: Explore (accepted papers from cleaned_dfs) ---
rl = cleaned_dfs.get("Reading List", pd.DataFrame())
if not rl.empty and "year" in rl.columns:
    display(rl["year"].value_counts().sort_index())
if not rl.empty and "journal" in rl.columns:
    display(rl["journal"].dropna().replace("", pd.NA).dropna().value_counts().head(10))
if not rl.empty and "database" in rl.columns:
    display(rl["database"].dropna().replace("", pd.NA).dropna().value_counts().head(10))
if not rl.empty and "journal_quartile" in rl.columns:
    display(rl["journal_quartile"].dropna().replace("", pd.NA).dropna().value_counts())
for name, df in cleaned_dfs.items():
    if name != "Reading List" and not df.empty:
        print(f"\n{name} ({len(df)} rows):")
        display(df.head(3))

year
2012     2
2013     5
2014     4
2015    10
2016     7
2017    21
2018    18
2019    18
2020    38
2021    34
2022    29
2023    39
2024    45
2025    13
2026     2
Name: count, dtype: int64

journal
LAK                                           40
IEEE Transactions on Learning Technologies    17
Computers & Education                         16
ArXiv                                         10
Applied Sciences                               9
Internet and Higher Education                  9
EDM                                            9
Knowledge-Based Systems                        8
IEEE Access                                    8
Computers in Human Behavior                    8
Name: count, dtype: int64

database
Google Scholar         195
ACM Digital Library     38
ISI Web of Science      27
IEEE Xplore             14
Scopus                  10
IEEE XPlore              1
Name: count, dtype: int64

journal_quartile
Q1 Journal              125
Conference               85
Q2 Journal               42
Q3+ Journal              18
ArXiv                    10
Book Chapter              4
Unreferenced Journal      1
Name: count, dtype: int64


REC (43 rows):


...

,Task,Context,Teaching Method,Student Performance Definition,Target,Students,Courses,Data sources,Features,Moment of Prediction,Preprocessing Details,Models,Assessment Strategy,Performance Metric: Best Model,Comments,Limitations,source_page_id,source_title,schema_name
0,Classification,Higher Education,E-Learning,Grade Range,"A, B, C and F",253 second and third year undergraduate students,"5 Classes in Web Design, Web Technologies and ...","LMS logs, Student Demographics, Academic Backg...","[Consistency, Following, Review Score, Attenda...",End of Course,Data was standardized and nans were imputted u...,"kNN, Random Forest, Logistic Regression, SVM, ...",Holdout Method (80/20),"{Accuracy: 0.826 - Stacking, Precision: 0.805 ...",Authors do a standard classification problem o...,Models overfit.\n\nPrediction of average grade...,02eedcf7-bc5a-406e-ba94-01f7e39a25cd,Enhancing educational evaluation through predi...,prediction_modeling
1,Classification,Higher Education,Blended Learning,End of Course Performance,Pass (>=5) vs Fail (<5),591 students,3 compulsory university courses: Physical Chem...,"LMS logs, Student characteristics, Learner grades",[Gender; forum views; page views; resource vie...,Monthly early prediction during semester; 6 cu...,Custom Moodle plugin exported cumulative month...,"Naïve Bayes, Random Forest, Bagging, PART, SMO...",10-fold Cross-Validation,Best AutoML results:\n{Accuracy: 0.9302 - Auto...,Main contribution is showing that AutoML impro...,Single institution and only 3 courses. Results...,02fef9f2-c5c2-47ee-a111-eb7788556c5e,Implementing AutoML in Educational Data Mining...,prediction_modeling
2,Regression,Higher Education,Blended Learning,End of Course Performance,inal grade (0–10),591 students,3 compulsory university courses: Physical Chem...,"LMS logs, Student characteristics, Learner grades",[Gender; forum views; page views; resource vie...,Monthly early prediction during semester; 6 cu...,Custom Moodle plugin exported cumulative month...,"Random Forest, M5Rules, Bagging, SMOreg, IBk-5...",10-fold Cross-Validation,Best AutoML results:\n{MAE = 1.0402 – M5P for ...,AutoML improved regression performance in all ...,Same limitations as above; results are course-...,02fef9f2-c5c2-47ee-a111-eb7788556c5e,Implementing AutoML in Educational Data Mining...,prediction_modeling



KT (63 rows):


,Task,Context,Teaching Method,Student Performance Definition,Target,Students,Courses,Data sources,Features,Flaw of Previous Models,Novelty of Model,Preprocessing Details,Models,Assessment Strategy,Performance Metric: Best Model,Comments,Limitations,source_page_id,source_title,schema_name
0,Classification,Intelligent Tutoring System,E-Learning,Predicting the Option Selected by Student in M...,Options Selected,10499 learners,Mathematics & German in Lernnavi,"Answer Sequences, Questions, Topic Taxonomies","[User Generated Transactions, Questions (and O...",Most approaches that are SoTA usually focus on...,The proposed model uses LLMs to understand con...,Users are first organized into learning sessio...,"MCQStudentBertCat, MCQStudentBertSum",Holdout Method,{MCC: 0.579 - MCQStudentBertCat with Mistral E...,Authors compare the 2 different models: MCQStu...,Limited interpretability of embedding-based mo...,17c20953-fb22-4346-b8e5-938e56b2be51,Student Answer Forecasting: Transformer-Driven...,sequence_tracing
1,Regression,Intelligent Tutoring System,,Score in Exercise,"Continuous score [0,1]","1825 Algebra05&06 learners,\n1112 Bridge Algeb...","Algebra, ASSISTments",Answer Sequences and Scores obtained,[Sequences of Scores for Previous Questions in...,Most of the current state-of-the-art KT models...,The addition of fuzziness to BKT allows for th...,"In essence, Fuzzy BKT and T2FBKT takes the per...","BKT, DKT, DKVMN, DKT+, DKT_DSC, DeepIRT, SAKT,...",5-Fold Cross-Validation,"{Average MAE Rank: 1.47 - T2FBKT, Average RMSE...",Authors explore the idea of introducing fuzine...,This fuzzy approach is exclusively applied to ...,18b4750c-f3ba-8001-846c-eddab66ea586,Fuzzy Bayesian Knowledge Tracing,sequence_tracing
2,Classification,MOOC,,Succeeding in Subsequent Programming Task,Succeed (1) vs Fail (0),"328128 HOC4 learners, 152869 HOC18 learners","HOC4, HOC8","Perfect Solution for each task, Intermediate l...","[pathLength, currentPerf, AIScore]","BKT, DKT or Additive Factor Model (AFM) strugg...",Authors present an Approaching Index - a measu...,Paths that do not lead to a solution have been...,"LR+pathLength, LR+currentPerf, PoissonPath, DK...",Holdout Method,HOC4 to HOC5\n{Accuracy: 0.95958 - LR+currentP...,"In programming tasks, reaching a solution can ...",Limited number of tasks for evaluation.\n\nPro...,18b4750c-f3ba-8005-9ff0-d7886becd1e4,Knowledge Tracing Within Single Programming Pr...,sequence_tracing



Summary Table (2 rows):


,Task,Context,Teaching Method,Student Performance Definition,Target,Students,Courses,Data sources,Features,Moment of Prediction,...,Limitations,source_page_id,source_title,schema_name,Recommender System Type,Target of Recommendation,Initialization Method,Updates to Recommendations,Recommendation types,Evaluation
0,Classification,MOOC,E-Learning,Obtaining Certification,Certificate Obtained (1) vs Certificate Not Ob...,338223 registrations,HarvardX courses,"Student Characteristics, \nLMS logs,\nPerforma...","[course_id, userid_DI, registered, viewed, exp...",End of Program,...,Data dependent: Co-Training requires a substan...,91c51bec-fe5f-4dfd-a61d-cdb8a204881e,Co-Training Method Based on Semi-Decoupling Fe...,Summary Table,,,,,,
1,,Guidelines for Educational Recommender Systems,E-Learning,,,125 educators\n595 students,- 1 Introduction to dotLRN course\n- 1 EBIFE,Learning Style Questionnaire,,,...,Paper merely describes practical guidelines. N...,b754f576-1ec5-46b9-ba21-b34febde24ec,Practical guidelines for designing and evaluat...,Summary Table,Ontology-Based,Actions to take in platform.,Learning Style Questionnaire\nProfile data,- No clear way of updating userbase automatica...,Next Action to take on the LMS platform.,Validated by Educators and Learners.\n\nLearni...
